# Install dependencies

In [ ]:
!pip install transformers torch pandas datasets

# Import libraries

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


# Load dataset, encode data, split train and test dataset, data training and testing


In [ ]:
# Load dataset
file_path = "/content/gdrive/MyDrive/Omdena/NLP in drug prediction/test/processed_diseases-583 (1).csv"
df = pd.read_csv(file_path)

# Ensure required columns exist
if "Symptoms" not in df.columns or "Disease" not in df.columns:
    raise ValueError("The dataset must contain 'Symptoms' (input) and 'Disease' (output) columns.")

# Encode diseases into numerical labels
label_encoder = LabelEncoder()
df["Label"] = label_encoder.fit_transform(df["Disease"])  # Convert disease names to numbers
num_classes = len(label_encoder.classes_)  # Count unique disease categories
print(f"Number of disease classes: {num_classes}")

# Split into 80% train, 20% test
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["Symptoms"].tolist(), df["Label"].tolist(), test_size=0.2, random_state=42
)

# Load tokenizer
model_name = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize input text
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512)

# Convert data to PyTorch Dataset (formats the data into PyTorch tensors, making it compatible with PyTorch's DataLoader for training deep learning models.)
# torch.utils.data.Dataset is a base class in PyTorch that allows you to create and load datasets for deep learning models.
class DiseaseDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self): #Define dataset size
        return len(self.labels)

    def __getitem__(self, idx): #Retrieve samples
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = DiseaseDataset(train_encodings, train_labels)
test_dataset = DiseaseDataset(test_encodings, test_labels)

# Load model with correct number of labels
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_classes)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,  # Adjust epochs based on dataset size
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# Train the model
trainer.train()

# Save the fine-tuned model
##save_path = "/content/gdrive/MyDrive/Omdena/NLP in drug prediction/test/fine_tuned_bio_clinicalbert"
#model.save_pretrained(save_path)
#tokenizer.save_pretrained(save_path)
#print(f"Fine-tuned model saved successfully at {save_path}!")

# Evaluate the model
trainer.evaluate()

# Save test dataset with actual and predicted values
def predict_disease(symptoms):
    inputs = tokenizer(symptoms, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    predicted_label = torch.argmax(outputs.logits, dim=1).item()
    predicted_disease = label_encoder.inverse_transform([predicted_label])[0]  # Convert number back to disease name
    return predicted_disease

df_test = pd.DataFrame({"Symptoms": test_texts, "Actual_Disease": label_encoder.inverse_transform(test_labels)})
df_test["Predicted_Disease"] = df_test["Symptoms"].apply(predict_disease)

# Save the results
test_output_path = "/content/gdrive/MyDrive/Omdena/NLP in drug prediction/test/test_predictions.csv"
df_test.to_csv(test_output_path, index=False)
print(f"Test predictions saved at {test_output_path}!")

# Display a sample of the predictions
df_test.head()


Number of disease classes: 1128


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

...

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yhlien1221 (yhlien1221-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,7.146500,7.283339
2,7.098200,7.480350
3,6.975500,7.739559
4,6.787100,7.785530
5,6.722900,7.795549


Fine-tuned model saved successfully at /content/gdrive/MyDrive/Omdena/NLP in drug prediction/test/fine_tuned_bio_clinicalbert!


Test predictions saved at /content/gdrive/MyDrive/Omdena/NLP in drug prediction/test/test_predictions.csv!


,Symptoms,Actual_Disease,Predicted_Disease
0,"Chorea, cognitive decline, mood swings, rigidity.",Huntington’s Disease,Upper Respiratory Infection (URI)
1,"Chronic itching, thickened skin, scratching ma...",Neurodermatitis,Upper Respiratory Infection (URI)
2,"Weight loss, palpitations, bulging eyes, heat ...",Graves’ Disease,Upper Respiratory Infection (URI)
3,"Fever, fatigue, weight loss, opportunistic inf...",HIV/AIDS,Upper Respiratory Infection (URI)
4,"Vision issues (infants), abnormal eye movements.",ROP (Retinopathy of Prematurity),Upper Respiratory Infection (URI)
